# FCN（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
セマンティックセグメンテーションの先駆けとなった手法であるFCN（Fully Convolutional Network）を，PASCAL VOC 2007データセットを用いて実装する．画像分類ネットワークの全結合層を畳み込み層に置き換える（Fully Convolutional化する）ことで，任意サイズの画像に対して画素単位のクラス分類を行う仕組みと，浅い層の特徴マップを利用して出力を高解像度化するスキップ接続について理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCSegmentation
from torchvision.models import vgg16, VGG16_Weights
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して画素単位のクラスラベル（セグメンテーションマスク）が付与されたデータセットで，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます（矩形領域のみのVOCDetectionと異なり，セグメンテーション用のアノテーションが存在する画像のみに限られるため，画像分類・物体検出のノートブックと比べて画像枚数が少なくなっています）．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．マスク画像は，各画素の値がクラスID（0が背景，1〜20が物体クラス，255が物体の境界など評価から除外する「無視領域」）になっている画像として与えられます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## FCN（Fully Convolutional Network）
FCNは，画像分類用のCNN（本ノートブックでは事前学習済みVGG16）が持つ全結合層を，1×1の畳み込み層に置き換えることで，画像全体に対して画素単位のクラス分類を行えるように拡張したネットワークです．全結合層は入力サイズが固定されますが，畳み込み層に置き換えることで任意サイズの画像を入力できるようになります．

### バックボーン（VGG16）
`torchvision_models.ipynb`で扱った事前学習済みVGG16の`features`部分を，畳み込みブロックの区切り（各ブロック末尾のMaxPool2d）ごとに3つに分割して使用します．最も深い出力は入力に対して1/32のサイズまでダウンサンプリングされており，そのままアップサンプリングすると輪郭がぼやけた粗いセグメンテーション結果になってしまいます．

In [ ]:
class FCN(nn.Module):
    """VGG16をバックボーンとしたFCN-8s"""
    def __init__(self, n_class=n_class):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.pool3 = vgg[:17]    # conv1_1 ... pool3, 出力stride 8,  channels 256
        self.pool4 = vgg[17:24]  # conv4_1 ... pool4, 出力stride 16, channels 512
        self.pool5 = vgg[24:31]  # conv5_1 ... pool5, 出力stride 32, channels 512

        # 全結合層を1x1・7x7の畳み込みに置き換えた部分（Fully Convolutional化）
        self.fc = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=7, padding=3), nn.ReLU(inplace=True), nn.Dropout2d(0.5),
            nn.Conv2d(4096, 4096, kernel_size=1), nn.ReLU(inplace=True), nn.Dropout2d(0.5),
        )

        self.score_fr = nn.Conv2d(4096, n_class, kernel_size=1)
        self.score_pool4 = nn.Conv2d(512, n_class, kernel_size=1)
        self.score_pool3 = nn.Conv2d(256, n_class, kernel_size=1)

        # 転置畳み込みによるアップサンプリング（学習可能なパラメータを持つ）
        self.upscore2 = nn.ConvTranspose2d(n_class, n_class, kernel_size=4, stride=2, padding=1)
        self.upscore_pool4 = nn.ConvTranspose2d(n_class, n_class, kernel_size=4, stride=2, padding=1)
        self.upscore8 = nn.ConvTranspose2d(n_class, n_class, kernel_size=16, stride=8, padding=4)

    def forward(self, x):
        p3 = self.pool3(x)   # stride 8
        p4 = self.pool4(p3)  # stride 16
        p5 = self.pool5(p4)  # stride 32

        h = self.fc(p5)
        h = self.score_fr(h)
        h = self.upscore2(h)  # stride 32 -> 16

        h = h + self.score_pool4(p4)  # skip connection：pool4の特徴を加算し，浅い層の情報を補う
        h = self.upscore_pool4(h)     # stride 16 -> 8

        h = h + self.score_pool3(p3)  # skip connection：pool3の特徴を加算し，さらに情報を補う
        h = self.upscore8(h)          # stride 8 -> 1（入力画像と同じ解像度）

        return h

### スキップ接続（Skip Connection）
`pool5`の出力だけをアップサンプリングした結果は，FCN-32sと呼ばれ，物体の輪郭が非常に粗くなります．そこで，より浅い層（`pool4`, `pool3`）のクラススコアを，アップサンプリング途中の出力に加算することで，高解像度な情報を補いながら最終的に入力画像と同じ解像度まで復元します（この構成をFCN-8sと呼びます）．

## 損失関数
セグメンテーションは，画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用できます．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
FCNを構築し，最適化手法としてAdamを設定します．バックボーンには事前学習済みの重みを使用しているため，スクラッチ学習の分類ノートブック群（`classification/`）よりも小さい学習率を用います．

In [ ]:
model = FCN(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてFCNを学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoU（Intersection over Union）の平均であるmIoU（mean IoU）を求めます．mIoUの算出には，`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## 検出結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します．

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのFCNとの違い

| 項目 | オリジナル論文 (Long et al., 2015) | 本ノートブック |
| :-- | :-- | :-- |
| 学習方法 | FCN-32s→FCN-16s→FCN-8sの順に段階的に学習（各段階を前段階の重みで初期化） | FCN-8sの構成を最初から通しで学習（段階的な学習は行わない） |
| 全結合層の初期化 | 分類用VGG16の全結合層（fc6, fc7）の重みを畳み込み層に転用して初期化 | ランダム初期化（畳み込み層への変換のみ行い，重みの転用は行わない） |
| データ拡張 | マルチスケール学習など複数の工夫を実施 | 固定サイズへのリサイズのみ（データ拡張なし） |
| 学習データ量 | PASCAL VOCの拡張データ（SBD）を含む約1万枚以上 | VOC2007のtrainvalのみ（422枚） |

## 課題

1. `epoch_num`や学習率を変更し，mIoUがどのように変化するか確認してください．
2. スキップ接続を使わない場合（`pool5`の出力のみをアップサンプリングするFCN-32s）を実装し，本ノートブックのFCN-8sと比較してください．
3. バックボーンをVGG16から別のモデル（例えば`torchvision.models.resnet50`）に変更した場合，実装をどのように変更する必要があるか考えてみてください．